# YOLOv8n COCO (tanpa melatih)
        **vision › detection** · notebook `yolov8n_coco_zeroshot`

        **Kapan dipakai:** Deteksi cepat real-time untuk kelas COCO dengan pustaka ultralytics.

        **Jebakan umum:** Lisensi AGPL-3.0 (periksa bila dipakai di luar riset). Hanya kelas COCO.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

**Data demo:** Penn-Fudan (170 gambar pejalan kaki, kotak + mask); 120 latih / 50 uji tetap. Metrik: **AP50** (kotak benar bila IoU ≥ 0.5). Bobot dibaca dari `pretrained/` (offline).

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
MODEL_KEY = "yolov8n_coco_zeroshot"
OUT = out_dir("vision", "detection", MODEL_KEY)
SEED = 42; set_seed(SEED)
SCORE_THR = 0.05                 # ambang skor deteksi (rendah untuk AP; naikkan untuk hasil bersih)
EPOCHS = 15

## 2. Data: gambar + kotak pejalan kaki (dari mask instance)
120 gambar pertama latih, 50 terakhir uji (tetap).

In [ ]:
from superdb.vision_det import load_pennfudan, split_idx, ap50, box_iou
D = load_pennfudan(); itr, ite = split_idx(len(D), 50)
imgs = [d["img"] for d in D]; boxes = [d["boxes"] for d in D]
print(len(D), "gambar | rata-rata orang per gambar:", round(float(np.mean([len(b) for b in boxes])), 2))
import matplotlib.patches as mp
fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
for a, i in zip(ax, (0, 1, 2)):
    a.imshow(imgs[i]); [a.add_patch(mp.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, color="lime", lw=2)) for b in boxes[i]]; a.axis("off")
plt.tight_layout(); plt.show()

## 3. YOLOv8n (ultralytics) dengan bobot lokal `pretrained/yolov8n/yolov8n.pt`

In [ ]:
from pathlib import Path
import torch
from ultralytics import YOLO
from superdb.paths import ROOT as R
WEIGHTS = R / "pretrained" / "yolov8n" / "yolov8n.pt"
assert WEIGHTS.exists(), "bobot belum ada: python tools/download_pretrained_vision.py yolov8n"
dev = 0 if torch.cuda.is_available() else "cpu"
def yolo_predict(m, ims, conf=SCORE_THR, cls=0):
    out = []
    for r in m.predict(source=[x[:, :, ::-1] for x in ims], conf=conf, device=dev, verbose=False, imgsz=416):    # ultralytics menganggap BGR
        k = r.boxes.cls.cpu().numpy() == cls; out.append((r.boxes.xyxy.cpu().numpy()[k], r.boxes.conf.cpu().numpy()[k]))
    return out

## 4. Deteksi tanpa melatih (kelas 0 = person pada COCO)

In [ ]:
model = YOLO(str(WEIGHTS)); preds = yolo_predict(model, [imgs[i] for i in ite])
metrics = ap50(preds, [boxes[i] for i in ite]); display(show_table(metrics))
import matplotlib.patches as mp
fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
for a, k in zip(ax, (0, 1, 2)):
    i = ite[k]; a.imshow(imgs[i]); a.axis("off"); b_, s_ = preds[k]; [a.add_patch(mp.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, color="red", lw=1.5)) for b, s in zip(b_, s_) if s > 0.4]
plt.tight_layout(); plt.savefig(OUT / "detections.png", dpi=130); plt.show()
save_json({"metrics": metrics}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Skor ambang rendah menghasilkan banyak kotak palsu; pilih ambang dari kurva precision-recall pada validation.
- Data lomba: ubah loader (`load_pennfudan`) menjadi pembaca anotasi lomba (COCO JSON/Pascal VOC/YOLO). Evaluasi `ap50` bisa dipakai apa adanya.
- Gambar besar dengan objek kecil: naikkan `imgsz`.